# Distributed Object Detection Training with Faster R-CNN

This example demonstrates how to train a [Faster R-CNN](https://arxiv.org/abs/1506.01497) object detection model using [PyTorch Distributed Data Parallel (DDP)](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html) and the [Penn-Fudan Pedestrian Detection](https://www.cis.upenn.edu/~jshi/ped_html/) dataset.

**What makes object detection different from image classification?**

- In classification, each image has a single label. In object detection, each image has a variable number of bounding boxes and labels.
- Because images have different numbers of objects, you cannot use PyTorch's default collate function (which tries to stack tensors). A custom `collate_fn` is required.
- Detection models like Faster R-CNN return a dictionary of losses in training mode, rather than raw logits.

This notebook walks you through running the example locally, and how to scale it across multiple nodes with Kubeflow TrainJob.

## Install the Kubeflow SDK

You need to install the Kubeflow SDK to interact with Kubeflow Trainer APIs:

In [ ]:
# !pip install -U kubeflow

## Install the PyTorch Dependencies

You also need to install PyTorch and Torchvision to run the example locally:

In [ ]:
!pip install torch==2.9.1
!pip install torchvision==0.22.1

## Define the Training Function

The training function downloads the Penn-Fudan Pedestrian dataset, builds a Faster R-CNN model with a MobileNet v3 backbone, and trains it using PyTorch DDP.

Key differences from image classification:
- A custom `Dataset` class loads images and parses XML/PNG mask annotations into bounding boxes
- A custom `collate_fn` handles variable-length targets (different images have different numbers of boxes)
- The model returns a dictionary of four loss components in training mode

In [ ]:
def train_object_detection():
    import os
    import pathlib
    import urllib.request
    import zipfile

    import torch
    import torch.distributed as dist
    import torch.utils.data
    import torchvision
    from torchvision import tv_tensors
    from torchvision.io import read_image
    from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchvision.transforms import v2 as T

    # --- Dataset class ---
    class PennFudanDataset(torch.utils.data.Dataset):
        """Penn-Fudan Pedestrian Detection and Segmentation dataset.

        Each sample returns (image, target) where target is a dict with:
        - boxes: FloatTensor[N, 4] in (x1, y1, x2, y2) format
        - labels: Int64Tensor[N]
        - masks: UInt8Tensor[N, H, W]
        - image_id: int
        - area: FloatTensor[N]
        - iscrowd: Int64Tensor[N]
        """

        def __init__(self, root, transforms=None):
            self.root = pathlib.Path(root)
            self.transforms = transforms
            # Sort to ensure reproducible ordering across ranks
            self.imgs = sorted((self.root / "PNGImages").glob("*.png"))
            self.masks = sorted((self.root / "PedMasks").glob("*.png"))

        def __getitem__(self, idx):
            img = read_image(str(self.imgs[idx]))
            mask = read_image(str(self.masks[idx]))

            # Instances are encoded as different colors in the mask
            obj_ids = torch.unique(mask)
            # Remove background (id 0)
            obj_ids = obj_ids[1:]
            num_objs = len(obj_ids)

            # Create binary masks for each instance
            masks = (mask == obj_ids[:, None, None]).to(dtype=torch.uint8)

            # Compute bounding boxes from masks
            boxes = torchvision.ops.masks_to_boxes(masks)

            # All instances are pedestrians (class 1)
            labels = torch.ones((num_objs,), dtype=torch.int64)

            image_id = idx
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
            iscrowd = torch.zeros((num_objs,), dtype=torch.int64)

            # Wrap in tv_tensors for transform compatibility
            img = tv_tensors.Image(img)
            target = {
                "boxes": tv_tensors.BoundingBoxes(
                    boxes, format="XYXY", canvas_size=img.shape[-2:]
                ),
                "masks": tv_tensors.Mask(masks),
                "labels": labels,
                "image_id": image_id,
                "area": area,
                "iscrowd": iscrowd,
            }

            if self.transforms is not None:
                img, target = self.transforms(img, target)

            return img, target

        def __len__(self):
            return len(self.imgs)

    # --- Transforms ---
    def get_transforms():
        return T.Compose([
            T.ToDtype(torch.float, scale=True),
            T.ToPureTensor(),
        ])

    # --- Custom collate function ---
    # Object detection targets have variable length (different number of boxes
    # per image), so the default collate (torch.stack) cannot be used.
    def collate_fn(batch):
        return tuple(zip(*batch))

    # --- Distributed setup ---
    device, backend = (
        ("cuda", "nccl") if torch.cuda.is_available() else ("cpu", "gloo")
    )
    print(f"Using Device: {device}, Backend: {backend}")

    local_rank = int(os.getenv("LOCAL_RANK", 0))
    dist.init_process_group(backend=backend)
    print(
        "Distributed Training for WORLD_SIZE: {}, RANK: {}, LOCAL_RANK: {}".format(
            dist.get_world_size(),
            dist.get_rank(),
            local_rank,
        )
    )

    # --- Download dataset (rank 0 only) ---
    data_dir = "./data"
    dataset_dir = os.path.join(data_dir, "PennFudanPed")
    if local_rank == 0:
        if not os.path.isdir(dataset_dir):
            os.makedirs(data_dir, exist_ok=True)
            zip_path = os.path.join(data_dir, "PennFudanPed.zip")
            print("Downloading Penn-Fudan dataset...")
            urllib.request.urlretrieve(
                "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip",
                zip_path,
            )
            print("Extracting dataset...")
            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(data_dir)
            os.remove(zip_path)
            print(f"Dataset ready at {dataset_dir}")
        else:
            print(f"Dataset already exists at {dataset_dir}")
    dist.barrier()

    # --- Create dataset and dataloader ---
    dataset = PennFudanDataset(dataset_dir, transforms=get_transforms())
    print(f"Dataset size: {len(dataset)} images")

    sampler = torch.utils.data.distributed.DistributedSampler(dataset)
    data_loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=2,
        sampler=sampler,
        collate_fn=collate_fn,
    )

    # --- Build model ---
    # Faster R-CNN with MobileNet v3 Large FPN backbone (pretrained on COCO)
    model = fasterrcnn_mobilenet_v3_large_fpn(weights="DEFAULT")
    # Replace the classifier head for our 2-class problem (background + pedestrian)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=2)

    device = torch.device(f"{device}:{local_rank}")
    model.to(device)
    model = torch.nn.parallel.DistributedDataParallel(model)

    # --- Optimizer ---
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(
        params, lr=0.005, momentum=0.9, weight_decay=0.0005
    )

    # --- Training loop ---
    num_epochs = 2
    for epoch in range(1, num_epochs + 1):
        model.train()
        sampler.set_epoch(epoch)
        epoch_loss = 0.0
        num_batches = 0

        for batch_idx, (images, targets) in enumerate(data_loader):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]

            # Faster R-CNN returns a dict of losses in training mode
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            epoch_loss += losses.item()
            num_batches += 1

            if batch_idx % 10 == 0 and dist.get_rank() == 0:
                # Print all four loss components
                loss_str = ", ".join(
                    f"{k}: {v.item():.4f}" for k, v in loss_dict.items()
                )
                print(
                    f"Epoch [{epoch}/{num_epochs}], "
                    f"Batch [{batch_idx}/{len(data_loader)}], "
                    f"Loss: {losses.item():.4f} ({loss_str})"
                )

        if dist.get_rank() == 0:
            avg_loss = epoch_loss / num_batches
            print(f"Epoch [{epoch}/{num_epochs}] Average Loss: {avg_loss:.4f}")

    # --- Cleanup ---
    dist.barrier()
    if dist.get_rank() == 0:
        print("Training is finished")
    dist.destroy_process_group()

## Run the Training Locally

We can submit the training function to the local Trainer client to run it in an isolated subprocess.

In [ ]:
from kubeflow.trainer import CustomTrainer, TrainerClient, LocalProcessBackendConfig

# Initialize local backend
backend_config = LocalProcessBackendConfig(cleanup_venv=True)
client = TrainerClient(backend_config=backend_config)

# List available runtimes
for runtime in client.list_runtimes():
    if runtime.name == "torch-distributed":
        torch_runtime = runtime
        break

# Submit training job
job_name = client.train(
    trainer=CustomTrainer(
        func=train_object_detection,
        packages_to_install=["torch", "torchvision"],
    ),
    runtime=torch_runtime,
)

# Stream logs
for logline in client.get_job_logs(job_name, follow=True):
    print(logline, end='')

## Scale PyTorch DDP with Kubeflow TrainJob

You can use `TrainerClient()` from the Kubeflow SDK to communicate with Kubeflow Trainer APIs and scale your training function across multiple PyTorch training nodes.

`TrainerClient()` verifies that you have required access to the Kubernetes cluster.

Kubeflow Trainer creates a `TrainJob` resource and automatically sets the appropriate environment variables to set up PyTorch in distributed environment.

In [ ]:
from kubeflow.trainer import CustomTrainer, TrainerClient

client = TrainerClient()

## List the Training Runtimes

You can get the list of available Training Runtimes to start your TrainJob.

Additionally, it might show available accelerator type and number of available resources.

In [ ]:
for runtime in client.list_runtimes():
    print(runtime)
    if runtime.name == "torch-distributed":
        torch_runtime = runtime

## Run the Distributed TrainJob

Kubeflow TrainJob will train the Faster R-CNN model on 2 PyTorch nodes. Each node downloads the Penn-Fudan dataset independently and processes its shard of the data.

In [ ]:
job_name = client.train(
    trainer=CustomTrainer(
        func=train_object_detection,
        # Set how many PyTorch nodes you want to use for distributed training.
        num_nodes=2,
        # Set the resources for each PyTorch node.
        resources_per_node={
            "cpu": 4,
            "memory": "8Gi",
            # Uncomment this to distribute the TrainJob using GPU nodes.
            # "nvidia.com/gpu": 1,
        },
    ),
    runtime=torch_runtime,
)

## Check the TrainJob Steps

You can check the components of the TrainJob that was created.

Since the TrainJob performs distributed training across 2 nodes, it generates 2 steps: `trainer-node-0` and `trainer-node-1`.

You can get the individual status for each of these steps.

In [ ]:
# Wait for the running status.
client.wait_for_job_status(name=job_name, status={"Running"})

In [ ]:
for c in client.get_job(name=job_name).steps:
    print(f"Step: {c.name}, Status: {c.status}, Devices: {c.device} x {c.device_count}\n")

## Watch the TrainJob Logs

We can use the `get_job_logs()` API to get the TrainJob logs.

Since we run training on 2 nodes, each PyTorch node processes approximately 170/2 = 85 images from the dataset.

You should see four loss components printed: `loss_classifier`, `loss_box_reg`, `loss_objectness`, and `loss_rpn_box_reg`.

In [ ]:
for logline in client.get_job_logs(job_name, follow=True):
    print(logline)

## Delete the TrainJob

When TrainJob is finished, you can delete the resource.

In [ ]:
# client.delete_job(job_name)